# Silver Transactions

Convert released Bronze transactions into trusted monitoring inputs.

Input: workspace.sentinel_bronze.transactions

Planned outputs:
- Accepted transactions.
- Quarantined records with validation reasons.

Begin by inspecting schema and amount precision.
No tables are written during this inspection.

In [ ]:
from pyspark.sql import functions as F

# Read the persisted Bronze table, not future staged batches.
bronze_table = "workspace.sentinel_bronze.transactions"
bronze = spark.table(bronze_table)

# Inspect actual types before defining Silver transformations.
bronze.printSchema()

# Labels must remain outside operational data.
assert "is_fraud" not in bronze.columns, (
    "Unexpected fraud labels in Bronze"
)

# Confirm which released batches are available.
display(
    bronze
    .groupBy("replay_batch")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy("replay_batch")
)

# Compare a wider decimal representation with the intended monetary type.
# A successful DECIMAL(18,2) cast alone does not prove that no rounding occurred.
amount_precision = bronze.select(
    F.expr("TRY_CAST(amt AS DECIMAL(38, 18))").alias("wide_amount"),
    F.expr("TRY_CAST(amt AS DECIMAL(18, 2))").alias("money_amount")
)

display(
    amount_precision.agg(
        F.count("*").alias("input_rows"),
        F.count_if(
            F.col("wide_amount").isNull()
            | F.col("money_amount").isNull()
        ).alias("unparseable_or_out_of_range_amounts"),
        F.count_if(
            F.col("wide_amount")
            != F.col("money_amount").cast("decimal(38,18)")
        ).alias("amounts_changed_at_two_decimal_places")
    )
)

### Inspect Bronze rescued data

Check whether Auto Loader populated `_rescued_data` in Bronze and inspect any
rescued JSON keys before deciding whether `_rescued_data` should trigger a
validation reason.

In [ ]:
# Summarize whether _rescued_data is populated in Bronze.
display(
    bronze.agg(
        F.count("*").alias("input_rows"),
        F.count_if(F.col("_rescued_data").isNotNull()).alias(
            "rows_with_non_null_rescued_data"
        ),
        F.count_if(
            F.col("_rescued_data").isNotNull()
            & (F.trim(F.col("_rescued_data")) != "")
        ).alias("rows_with_non_blank_rescued_data")
    )
)

# Inspect distinct key combinations if any rescued JSON payloads exist.
rescued_key_summary = (
    bronze
    .filter(
        F.col("_rescued_data").isNotNull()
        & (F.trim(F.col("_rescued_data")) != "")
    )
    .select(
        F.array_sort(
            F.map_keys(
                F.from_json(F.col("_rescued_data"), "MAP<STRING,STRING>")
            )
        ).alias("rescued_keys")
    )
    .groupBy("rescued_keys")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.col("row_count").desc())
)

display(rescued_key_summary)

### Normalize required fields and convert types

- Normalize blank required identifiers (`trans_num`, `cc_num`, `merchant`) to `NULL`.
- Parse `trans_date_trans_time` as `TIMESTAMP_NTZ` (`event_time`) without assuming a timezone.
- Convert `amt` to `DECIMAL(18,2)` (`amount`) alongside `DECIMAL(38,18)` (`_wide_amount`).
- Preserve raw source values and lineage columns for investigation.

In [ ]:
# Identify lineage metadata columns so we can build a canonical signature
# across all source business attributes for duplicate comparison.
lineage_columns = {
    "source_file",
    "source_row_index",
    "replay_batch",
    "_rescued_data",
    "_ingested_file_path",
    "_ingestion_timestamp",
}

business_columns = sorted([
    col_name for col_name in bronze.columns
    if col_name not in lineage_columns
])

# Preserve raw values needed for quarantine investigation before normalizing.
typed_transactions = (
    bronze
    .withColumn("raw_trans_num", F.col("trans_num"))
    .withColumn("raw_cc_num", F.col("cc_num"))
    .withColumn("raw_merchant", F.col("merchant"))
    .withColumn("raw_amt", F.col("amt"))
    .withColumn("raw_trans_date_trans_time", F.col("trans_date_trans_time"))
    .withColumn("trans_num", F.expr("NULLIF(TRIM(trans_num), '')"))
    .withColumn("cc_num", F.expr("NULLIF(TRIM(cc_num), '')"))
    .withColumn("merchant", F.expr("NULLIF(TRIM(merchant), '')"))
    .withColumn(
        "_normalized_event_time_str",
        F.expr("NULLIF(TRIM(trans_date_trans_time), '')")
    )
    .withColumn(
        "event_time",
        F.expr("TRY_CAST(_normalized_event_time_str AS TIMESTAMP_NTZ)")
    )
    .withColumn(
        "_normalized_amt_str",
        F.expr("NULLIF(TRIM(amt), '')")
    )
    .withColumn(
        "_wide_amount",
        F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(38, 18))")
    )
    .withColumn(
        "amount",
        F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(18, 2))")
    )
)

# Build a deterministic JSON signature over all source business columns.
# Using normalized strings (with an explicit sentinel for NULL) ensures
# that invalid rows still participate in conflict detection.
business_signature_struct = F.struct(*[
    F.coalesce(
        F.expr(f"NULLIF(TRIM(`{col_name}`), '')").cast("string"),
        F.lit("__NULL__")
    ).alias(col_name)
    for col_name in business_columns
])

typed_transactions = typed_transactions.withColumn(
    "_business_signature",
    F.to_json(business_signature_struct)
)

### Assign field validation reasons

Evaluate every required identifier, event timestamp, and monetary amount check.
Preserve all applicable reasons on each row.

In [ ]:
raw_validation_reasons = F.array(
    F.when(
        F.col("trans_num").isNull(),
        F.lit("missing_transaction_id")
    ),
    F.when(
        F.col("cc_num").isNull(),
        F.lit("missing_card_id")
    ),
    F.when(
        F.col("merchant").isNull(),
        F.lit("missing_merchant")
    ),
    F.when(
        F.col("_normalized_event_time_str").isNull(),
        F.lit("missing_event_time")
    ),
    F.when(
        F.col("_normalized_event_time_str").isNotNull()
        & F.col("event_time").isNull(),
        F.lit("unparseable_event_time")
    ),
    F.when(
        F.col("_normalized_amt_str").isNull(),
        F.lit("missing_amount")
    ),
    F.when(
        F.col("_normalized_amt_str").isNotNull()
        & (F.col("_wide_amount").isNull() | F.col("amount").isNull()),
        F.lit("unparseable_or_out_of_range_amount")
    ),
    F.when(
        F.col("_wide_amount").isNotNull()
        & F.col("amount").isNotNull()
        & (
            F.col("_wide_amount")
            != F.col("amount").cast("decimal(38,18)")
        ),
        F.lit("amount_precision_changed")
    ),
    F.when(
        F.col("_wide_amount").isNotNull()
        & (F.col("_wide_amount") <= F.lit(0).cast("decimal(38,18)")),
        F.lit("non_positive_amount")
    )
)

validated_transactions = typed_transactions.withColumn(
    "field_validation_reasons",
    F.filter(raw_validation_reasons, lambda reason: reason.isNotNull())
)

### Classify duplicate transaction IDs across Bronze

- Check `trans_num` multiplicity across the entire Bronze input before filtering invalid rows.
- Distinguish identical business records (`distinct_business_signatures == 1`) from conflicting records (`distinct_business_signatures > 1`).
- Quarantine all rows involved in a conflicting duplicate ID without selecting an arbitrary winner.
- For valid identical duplicates, retain one deterministic copy as `accepted` and classify extra copies as `duplicate_identical`.

In [ ]:
from pyspark.sql.window import Window

# Summarize occurrence count and distinct business payloads per non-null ID
# across all Bronze rows, including rows with field validation errors.
duplicate_summary = (
    validated_transactions
    .filter(F.col("trans_num").isNotNull())
    .groupBy("trans_num")
    .agg(
        F.count("*").alias("id_occurrence_count"),
        F.countDistinct("_business_signature").alias(
            "distinct_business_signatures"
        )
    )
)

# Order occurrences deterministically to retain a single copy only when
# duplicate records have identical business attributes.
id_order_window = Window.partitionBy(
    F.when(F.col("trans_num").isNotNull(), F.col("trans_num"))
    .otherwise(F.monotonically_increasing_id().cast("string"))
).orderBy(
    F.col("_ingestion_timestamp").asc_nulls_last(),
    F.col("_ingested_file_path").asc_nulls_last(),
    F.expr("TRY_CAST(source_row_index AS BIGINT)").asc_nulls_last(),
    F.col("source_row_index").asc_nulls_last()
)

classified_transactions = (
    validated_transactions
    .join(duplicate_summary, on="trans_num", how="left")
    .withColumn(
        "duplicate_rank",
        F.when(
            F.col("trans_num").isNotNull(),
            F.row_number().over(id_order_window)
        )
    )
    .withColumn(
        "duplicate_status",
        F.when(F.col("trans_num").isNull(), F.lit("missing_id"))
        .when(F.col("id_occurrence_count") == 1, F.lit("unique_id"))
        .when(
            (F.col("id_occurrence_count") > 1)
            & (F.col("distinct_business_signatures") > 1),
            F.lit("conflicting_duplicate")
        )
        .when(
            (F.col("id_occurrence_count") > 1)
            & (F.col("distinct_business_signatures") == 1)
            & (F.col("duplicate_rank") == 1),
            F.lit("identical_duplicate_primary")
        )
        .otherwise(F.lit("identical_duplicate_extra"))
    )
    .withColumn(
        "duplicate_validation_reasons",
        F.when(
            F.col("duplicate_status") == "conflicting_duplicate",
            F.array(F.lit("conflicting_duplicate_transaction_id"))
        )
        .when(
            F.col("duplicate_status") == "identical_duplicate_extra",
            F.array(F.lit("duplicate_identical_transaction_id"))
        )
        .otherwise(F.array().cast("array<string>"))
    )
    .withColumn(
        "validation_reasons",
        F.concat(
            F.col("field_validation_reasons"),
            F.col("duplicate_validation_reasons")
        )
    )
    .withColumn(
        "record_disposition",
        F.when(
            F.size(F.col("validation_reasons")) == 0,
            F.lit("accepted")
        )
        .when(
            (F.col("duplicate_status") == "identical_duplicate_extra")
            & (F.size(F.col("field_validation_reasons")) == 0),
            F.lit("duplicate_identical")
        )
        .otherwise(F.lit("quarantined"))
    )
)

# Separate intermediate DataFrames by disposition (no tables written yet).
accepted_transactions = classified_transactions.filter(
    F.col("record_disposition") == "accepted"
)
quarantined_records = classified_transactions.filter(
    F.col("record_disposition") == "quarantined"
)
duplicate_identical_records = classified_transactions.filter(
    F.col("record_disposition") == "duplicate_identical"
)

### Reconcile rows and inspect validation results

Verify that every Bronze input row is accounted for exactly once and that
accepted transaction IDs are unique.

In [ ]:
reconciliation_metrics = classified_transactions.agg(
    F.count("*").alias("classified_rows"),
    F.count_if(F.col("record_disposition").isNull()).alias(
        "unclassified_rows"
    ),
    F.count_if(F.col("record_disposition") == "accepted").alias(
        "accepted_rows"
    ),
    F.count_if(F.col("record_disposition") == "quarantined").alias(
        "quarantined_rows"
    ),
    F.count_if(F.col("record_disposition") == "duplicate_identical").alias(
        "duplicate_identical_rows"
    ),
    F.count_if(
        (F.col("record_disposition") == "accepted")
        & (
            F.col("trans_num").isNull()
            | (F.size(F.col("validation_reasons")) > 0)
        )
    ).alias("invalid_accepted_rows"),
    F.count_if(
        (F.col("record_disposition") != "accepted")
        & (F.size(F.col("validation_reasons")) == 0)
    ).alias("non_accepted_without_reasons"),
    F.count_if(F.col("_rescued_data").isNotNull()).alias(
        "rows_with_rescued_data"
    )
).collect()[0]

bronze_input_rows = bronze.count()
distinct_accepted_ids = (
    accepted_transactions.select("trans_num").distinct().count()
)

assert reconciliation_metrics["classified_rows"] == bronze_input_rows, (
    "Classified row count does not match Bronze input count"
)
assert reconciliation_metrics["unclassified_rows"] == 0, (
    "Every input row must receive a disposition"
)
assert (
    reconciliation_metrics["accepted_rows"]
    + reconciliation_metrics["quarantined_rows"]
    + reconciliation_metrics["duplicate_identical_rows"]
    == bronze_input_rows
), "Disposition counts do not sum to Bronze input count"
assert reconciliation_metrics["invalid_accepted_rows"] == 0, (
    "Accepted rows must have a non-null trans_num and no validation reasons"
)
assert reconciliation_metrics["non_accepted_without_reasons"] == 0, (
    "Every quarantined or duplicate row must record a validation reason"
)
assert distinct_accepted_ids == reconciliation_metrics["accepted_rows"], (
    "Accepted transactions contain duplicate transaction IDs"
)

print(f"Bronze input rows:        {bronze_input_rows:,}")
print(f"Accepted rows:            {reconciliation_metrics['accepted_rows']:,}")
print(f"Quarantined rows:         {reconciliation_metrics['quarantined_rows']:,}")
print(
    "Duplicate identical rows: "
    f"{reconciliation_metrics['duplicate_identical_rows']:,}"
)
print(f"Distinct accepted IDs:    {distinct_accepted_ids:,}")

# Display batch-level reconciliation summary.
display(
    classified_transactions
    .groupBy("replay_batch", "record_disposition", "duplicate_status")
    .agg(F.count("*").alias("row_count"))
    .orderBy("replay_batch", "record_disposition", "duplicate_status")
)

# Display validation reason frequencies across non-accepted records.
validation_reason_summary = (
    classified_transactions
    .filter(F.size(F.col("validation_reasons")) > 0)
    .select(
        "replay_batch",
        "record_disposition",
        F.explode("validation_reasons").alias("validation_reason")
    )
    .groupBy("replay_batch", "record_disposition", "validation_reason")
    .agg(F.count("*").alias("row_count"))
    .orderBy("replay_batch", "record_disposition", "validation_reason")
)

display(validation_reason_summary)